# Step 2: Data resampling

From the parsed CWRU bearing data (step 1), we resample each data file (csv) to the same sampling frequency

The specs follow:
- https://www.sciencedirect.com/science/article/pii/S0888327021010499
- https://arxiv.org/abs/2407.14625

The resulting data is called the 'resampled' data.


In [15]:
# Load the "autoreload" extension so that code can change
%load_ext autoreload
# Always reload modules so that as you change code in src, it gets loaded
%autoreload 2

import os
from copy import deepcopy
import json
import yaml
import re

import math
import numpy as np
import pandas as pd
import scipy
from scipy.signal import ShortTimeFFT
import librosa

import matplotlib.pyplot as plt



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load metadata

In [2]:
parsed_metadata_path = './data_parsed/parsed_metadata.json'
with open(parsed_metadata_path, 'r') as f:
    parsed_metadata = json.load(f)

In [3]:
len(parsed_metadata)

153

In [79]:
df_parsed_metadata = pd.DataFrame(parsed_metadata)

In [81]:
TOTAL_DURATION_SEC = 10.0
df_parsed_metadata['inferred_sampling_freq_hz'] = df_parsed_metadata['df_length'].apply(
    lambda x: math.floor(x / (TOTAL_DURATION_SEC * 1000)) * 1000)

In [83]:
df_parsed_metadata

,raw_data_path,cycle_id,fault_label,motor_load_hp,sampling_freq_khz,fault_type,fault_location,fault_diameter,OR_position,rpm,df_length,df_columns,parsed_data_path,inferred_sampling_freq_hz
0,./data/Normal/99_Normal_2.mat,99,N,2,12.0,N,None,None,None,1750.0,485063,"[DE_time, FE_time, cycle_id]",./data_parsed/99.csv,48000
1,./data/Normal/98_Normal_1.mat,98,N,1,12.0,N,None,None,None,1772.0,483903,"[DE_time, FE_time, cycle_id]",./data_parsed/98.csv,48000
2,./data/Normal/97_Normal_0.mat,97,N,0,12.0,N,None,None,None,1797.0,243938,"[DE_time, FE_time, cycle_id]",./data_parsed/97.csv,24000
3,./data/Normal/100_Normal_3.mat,100,N,3,12.0,N,None,None,None,1730.0,485643,"[DE_time, FE_time, cycle_id]",./data_parsed/100.csv,48000
4,./data/12k_Drive_End_Bearing_Fault_Data/B/007/...,121,DE_B,3,12.0,DE,B,007,None,1722.0,121556,"[DE_time, FE_time, BA_time, cycle_id]",./data_parsed/121.csv,12000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148,./data/48k_Drive_End_Bearing_Fault_Data/OR/021...,240,DE_OR,2,48.0,DE,OR,021,@6,1747.0,487964,"[DE_time, FE_time, cycle_id]",./data_parsed/240.csv,48000
149,./data/48k_Drive_End_Bearing_Fault_Data/OR/021...,264,DE_OR,2,48.0,DE,OR,021,@12,1746.0,486804,"[DE_time, FE_time, cycle_id]",./data_parsed/264.csv,48000
150,./data/48k_Drive_End_Bearing_Fault_Data/OR/021...,265,DE_OR,3,48.0,DE,OR,021,@12,1718.0,486224,"[DE_time, FE_time, cycle_id]",./data_parsed/265.csv,48000
151,./data/48k_Drive_End_Bearing_Fault_Data/OR/021...,263,DE_OR,1,48.0,DE,OR,021,@12,1771.0,486224,"[DE_time, FE_time, cycle_id]",./data_parsed/263.csv,48000


In [86]:
df_parsed_metadata[df_parsed_metadata['sampling_freq_khz']==12.0][['cycle_id', 'inferred_sampling_freq_hz']].sort_values('inferred_sampling_freq_hz')

,cycle_id,inferred_sampling_freq_hz
52,235,12000
75,279,12000
74,280,12000
73,278,12000
72,281,12000
...,...,...
39,146,12000
2,97,24000
3,100,48000
1,98,48000


In [26]:
df1 = pd.read_csv('./data_parsed/99.csv')

In [36]:
df1['timestamp'] = df1['index'].apply(lambda x: x / 48000)
df1 = df1.drop('index', axis=1)

In [37]:
df1

,DE_time,FE_time,BA_time,cycle_id,timestamp
0,0.064254,0.038625,NaN,99,0.000000
1,0.063002,0.096769,NaN,99,0.000021
2,-0.004381,0.127382,NaN,99,0.000042
3,-0.035882,0.144640,NaN,99,0.000063
4,-0.023991,0.086702,NaN,99,0.000083
...,...,...,...,...,...
485058,0.023991,-0.046022,NaN,99,10.105375
485059,0.034004,-0.027736,NaN,99,10.105396
485060,0.005215,0.031640,NaN,99,10.105417
485061,-0.065714,0.113000,NaN,99,10.105438


In [54]:
1/ df1['timestamp'].diff()

0                  NaN
1         48000.000000
2         48000.000000
3         48000.000000
4         48000.000000
              ...     
485058    47999.999998
485059    48000.000002
485060    47999.999998
485061    47999.999998
485062    48000.000002
Name: timestamp, Length: 485063, dtype: float64

In [47]:
df1['timestamp'].max() / 12000

0.0008421215277777778

In [69]:
int(len(df1) / 48000 * 12000)

121265.75

In [59]:
12000 * len(df1) * (1/ df1['timestamp'].diff().max())

279396287990725.0

In [70]:
x1, t1 = scipy.signal.resample(x=df1.drop(['timestamp', 'cycle_id'], axis=1).to_numpy(),
                      t=df1['timestamp'].to_numpy(),
                      num=int(len(df1) / 48000 * 12000))

In [87]:
t1

array([0.00000000e+00, 8.33338487e-05, 1.66667697e-04, ...,
       1.01052292e+01, 1.01053125e+01, 1.01053958e+01])

In [96]:
np.hstack([np.expand_dims(t1, axis=-1), x1]).shape

(121265, 4)

In [98]:
df2 = pd.DataFrame(np.hstack([np.expand_dims(t1, axis=-1), x1]))
df2

,0,1,2,3
0,0.000000,-0.018271,0.118596,NaN
1,0.000083,0.010231,0.070673,NaN
2,0.000167,-0.016071,-0.060309,NaN
3,0.000250,-0.000838,0.061054,NaN
4,0.000333,0.058258,0.100334,NaN
...,...,...,...,...
121260,10.105062,0.028959,0.073193,NaN
121261,10.105146,0.040522,0.008377,NaN
121262,10.105229,0.017982,0.165515,NaN
121263,10.105312,0.015304,-0.004521,NaN


In [71]:
1 / pd.Series(t1).diff()

0                  NaN
1         11999.925783
2         11999.925783
3         11999.925783
4         11999.925783
              ...     
121260    11999.925783
121261    11999.925783
121262    11999.925783
121263    11999.925783
121264    11999.925783
Length: 121265, dtype: float64

In [56]:
num=12000 * len(df1) * (1/ df1['timestamp'].diff().max())